In [2]:
import pandas as pd

from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


In [5]:
pip install langchain faiss-cpu sentence-transformers pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import langchain_community
print(langchain_community.__version__)


0.4.1


In [3]:
csv_path = r"D:\OVGU\Digital Engineering\Semester 3\Marcus Project\Data for Vector store.csv"
df = pd.read_csv(csv_path)

# Normalize labels (important)
df["label"] = df["label"].str.upper()


In [4]:
documents = []

for _, row in df.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={"label": row["label"]}
        )
    )


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


C:\Users\suraj\AppData\Local\Temp\ipykernel_8668\614204990.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\suraj\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
vectorstore = FAISS.from_documents(documents, embeddings)


In [7]:
vectorstore.save_local("hotel_review_vectorstore")


In [8]:
vectorstore = FAISS.load_local(
    "hotel_review_vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [9]:
query = "Absolutely perfect hotel, best stay ever!!!"

docs = retriever.invoke(query)

for d in docs:
    print("TEXT:", d.page_content)
    print("LABEL:", d.metadata["label"])
    print("-" * 40)


TEXT: Excellent Hotel ! Rooms and service were great. A great value ! I have been to places that are not as good for nearly twice the price. Good location. Close to the L and short walk to rental car service. Restaurants in the area a bit of a disappointment, but the shopping was great. Area recommendations Great theatre, restaurants, museums and shopping.
LABEL: TRUTHFUL
----------------------------------------
TEXT: This hotel is absolutely beautiful. I is elegant, comfortable and a very relaxing stay. It was easy to get to the airport and very close to shopping. I felt like a king in a castle surrounded by luxury and beauty. I would recommend this hotel to anyone.
LABEL: DECEPTIVE
----------------------------------------
TEXT: This hotel is in a great city with great personnel working there. The rooms are more than you can expect for a hotel. The services are abundant, leaving no room to complain. With their fine diners, you cannot go wrong with enjoying a peaceful meal while relaxi

In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}   # number of retrieved examples
)


In [11]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["examples", "review"],
    template="""
You are an expert at detecting fake hotel reviews.

Below are examples of hotel reviews with their labels.

{examples}

Now classify the following review as either TRUTHFUL or DECEPTIVE.

Review:
"{review}"

Respond with ONLY one word: TRUTHFUL or DECEPTIVE.
"""
)


In [12]:
def format_examples(docs):
    formatted = []
    for d in docs:
        formatted.append(
            f'Review: "{d.page_content}"\nLabel: {d.metadata["label"]}'
        )
    return "\n\n".join(formatted)


In [13]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="gemma:2b")


In [14]:
def normalize_prediction(response):
    response = response.upper()

    if "TRUTHFUL" in response:
        return "TRUTHFUL"
    elif "DECEPTIVE" in response:
        return "DECEPTIVE"
    else:
        return "UNKNOWN"


In [15]:
def classify_review_rag(review_text):
    # Retrieve similar labeled reviews
    docs = retriever.invoke(review_text)

    # Format them as few-shot examples
    examples = format_examples(docs)

    # Build prompt
    prompt = rag_prompt.format(
        examples=examples,
        review=review_text
    )

    # Call Gemma
    response = llm.invoke(prompt)

    # Clean output
    #label = response.strip().upper()
    #label = label.split()[0].strip(".,!?:;")
    label = normalize_prediction(response)

    return label


In [16]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix



In [17]:
csv_path = r"D:\OVGU\Digital Engineering\Semester 3\Marcus Project\sampled_deceptive_opinion_new.csv"

df = pd.read_csv(csv_path)

# Normalize labels
df["label"] = df["label"].str.upper().str.strip()

df.head()


,label,hotel,source,text
0,TRUTHFUL,james,Web,"I have stayed at the James a few times, it is ..."
1,TRUTHFUL,hilton,Web,The hotel is very impressive upon entering and...
2,TRUTHFUL,omni,TripAdvisor,My wife and I travelled to Chicago and really ...
3,TRUTHFUL,homewood,Web,After reading good reviews about I booked a 5 ...
4,TRUTHFUL,fairmont,TripAdvisor,We recently completed our second stay at the F...


In [18]:
y_true = []
y_pred = []

for idx, row in df.iterrows():
    review = row["text"]
    true_label = row["label"]

    pred_label = classify_review_rag(review)

    y_true.append(true_label)
    y_pred.append(pred_label)

    print(f"[{idx}] True: {true_label} | Pred: {pred_label}")


[0] True: TRUTHFUL | Pred: DECEPTIVE
[1] True: TRUTHFUL | Pred: DECEPTIVE
[2] True: TRUTHFUL | Pred: TRUTHFUL
[3] True: TRUTHFUL | Pred: DECEPTIVE
[4] True: TRUTHFUL | Pred: TRUTHFUL
[5] True: TRUTHFUL | Pred: DECEPTIVE
[6] True: TRUTHFUL | Pred: DECEPTIVE
[7] True: TRUTHFUL | Pred: DECEPTIVE
[8] True: TRUTHFUL | Pred: DECEPTIVE
[9] True: TRUTHFUL | Pred: DECEPTIVE
[10] True: TRUTHFUL | Pred: TRUTHFUL
[11] True: TRUTHFUL | Pred: TRUTHFUL
[12] True: TRUTHFUL | Pred: DECEPTIVE
[13] True: TRUTHFUL | Pred: TRUTHFUL
[14] True: TRUTHFUL | Pred: TRUTHFUL
[15] True: TRUTHFUL | Pred: TRUTHFUL
[16] True: TRUTHFUL | Pred: TRUTHFUL
[17] True: TRUTHFUL | Pred: DECEPTIVE
[18] True: TRUTHFUL | Pred: DECEPTIVE
[19] True: TRUTHFUL | Pred: DECEPTIVE
[20] True: TRUTHFUL | Pred: DECEPTIVE
[21] True: TRUTHFUL | Pred: DECEPTIVE
[22] True: TRUTHFUL | Pred: DECEPTIVE
[23] True: TRUTHFUL | Pred: DECEPTIVE
[24] True: TRUTHFUL | Pred: TRUTHFUL
[25] True: TRUTHFUL | Pred: DECEPTIVE
[26] True: TRUTHFUL | Pred: TRU

In [19]:
accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label="DECEPTIVE")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.4583
F1 Score: 0.4248


In [20]:
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["TRUTHFUL", "DECEPTIVE"]
)

print(cm)


[[62 58]
 [72 48]]


In [24]:
test_review = "My husband and I stayed here for New Years eve weekend. This is an excellent hotel in an excellent location. Hotel is very tastefully decorated and warmly welcoming. The rooms had the best electronics I've ever seen. Loved that they had a soaking tub and a spacious shower stall. The hotel restaurant seemed to be empty everytime we walked by, so we avoided it too. Ate at Joe's across the street - excellent seafood. And you can shop til you drop without going outside! Nordstroms and four story mall is attached. Lots of good 'food on the run' places in there. Loved Chicago - loved the Conrad. I will go back!"

result = classify_review_rag(test_review)
print("Predicted label:", result)


Predicted label: TRUTHFUL


In [7]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
   ---------------------------------------- 0.0/110.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/110.9 MB 15.9 MB/s eta 0:00:07
   ---------------------------------------- 0.7/110.9 MB 8.4 MB/s eta 0:00:14
    --------------------------------------- 1.6/110.9 MB 12.7 MB/s eta 0:00:09
    --------------------------------------- 2.1/110.9 MB 12.3 MB/s eta 0:00:09
    --------------------------------------- 2.7/110.9 MB 12.3 MB/s eta 0:00:09
   - -------------------------------------- 2.9/110.9 MB 12.5 MB/s eta 0:00:09
   - -------------------------------------- 3.7/110.9 MB 12.4 MB/s eta 0:00:09
   - -------------------------------------- 4.3/110.9 MB 12.4 MB/s eta 0:00:09
   - -------------------------------------- 4.4/110.9 MB 11.2 MB/s eta 0:00:10
   - -------------------------------------- 5.2/110.9 MB 11.8 MB/s eta 0:00:09
   - -------------------------------------- 5.4/110.9 MB 11.1 MB/s eta 0:00:10
   -

In [8]:
pip install sentence-transformers


   ---------------------------------------- 0.0/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
    --------------------------------------- 10.2/493.7 kB ? eta -:--:--
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   --- ----------------------------------- 41.0/493.7 kB 281.8 kB/s eta 0:00:02
   ----------- -------------------------- 143.4/493.7 kB 500.5 kB/s eta 0:00:01
   ---------------------------------- ----- 419.8/493.7 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 493.7/493.7 kB 1.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # should be False (this is OK)


2.9.1+cpu
False


In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully")


Model loaded successfully


In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded successfully")


c:\Users\suraj\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded successfully
